In [1]:
import os
import random
from PIL import Image
import torch
from torch.utils.data import Dataset
import torchvision.transforms.functional as TF

class SRImplicitDataset(Dataset):
    def __init__(self, img_dir, max_images=100):
        self.files = sorted([
            os.path.join(img_dir, f)
            for f in os.listdir(img_dir)
            if f.endswith(".png") or f.endswith(".jpg")
        ])[:max_images]

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img = Image.open(self.files[idx]).convert("RGB")
        hr = TF.to_tensor(img)

        h, w = hr.shape[1:]

        # HARD CLAMP HR (NO SKIP)
        h = max(2, h)
        w = max(2, w)
        hr = TF.resize(hr, (h, w), antialias=True)

        scale = random.uniform(1.5, 4.0)

        # SAFE LR (MIN = 2)
        lr_h = max(2, int(h / scale))
        lr_w = max(2, int(w / scale))

        lr = TF.resize(hr, (lr_h, lr_w), antialias=True)

        return lr, hr

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SRNOInspired(nn.Module):
    def __init__(self, hidden=256):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, hidden, 3, padding=1),
            nn.ReLU()
        )

        self.mlp = nn.Sequential(
            nn.Linear(hidden + 2, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 3)
        )

    def forward(self, lr, out_h, out_w):
        B, _, H_lr, W_lr = lr.shape

        if H_lr < 2 or W_lr < 2:
            lr = F.interpolate(lr, size=(max(2, H_lr), max(2, W_lr)), mode="bilinear", align_corners=False)

        feat = self.encoder(lr)

        B, C, H, W = feat.shape

        if H < 2 or W < 2:
            feat = F.interpolate(feat, size=(max(2, H), max(2, W)), mode="bilinear", align_corners=False)
            B, C, H, W = feat.shape

        out_h = max(2, out_h)
        out_w = max(2, out_w)

        y = torch.linspace(-1, 1, out_h, device=lr.device)
        x = torch.linspace(-1, 1, out_w, device=lr.device)
        yy, xx = torch.meshgrid(y, x, indexing="ij")

        grid = torch.stack((xx, yy), dim=-1)
        grid = grid.unsqueeze(0).repeat(B, 1, 1, 1)

        feat_up = F.grid_sample(
            feat,
            grid,
            mode='bilinear',
            align_corners=True,
            padding_mode='border'
        )

        feat_flat = feat_up.permute(0, 2, 3, 1).reshape(-1, C)
        coords_flat = grid.reshape(-1, 2)

        inp = torch.cat([feat_flat, coords_flat], dim=1)

        chunk_size = 50000
        outputs = []

        for i in range(0, inp.shape[0], chunk_size):
            chunk = inp[i:i+chunk_size]
            out_chunk = self.mlp(chunk)
            outputs.append(out_chunk)

        out = torch.cat(outputs, dim=0)

        del feat_flat, coords_flat, inp, outputs
        torch.cuda.empty_cache()

        out = out.view(B, out_h, out_w, 3).permute(0, 3, 1, 2)

        return out

In [3]:
# Load Dataset FRCNN
import torch
import numpy as np
import cv2
import os
from PIL import Image
import torchvision.transforms.functional as TF

from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

device = "cuda" if torch.cuda.is_available() else "cpu"

# -----------------------------
# LOAD FASTER R-CNN DETECTOR
# -----------------------------
NUM_CLASSES = 11  # adjust if needed (background + classes)

detector = fasterrcnn_mobilenet_v3_large_fpn(weights=None)


in_features = detector.roi_heads.box_predictor.cls_score.in_features
detector.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)

detector.load_state_dict(torch.load(
    "C:/Users/Mardyson Justin/Thesis/FCNN3/checkpoints/best_model.pth",
    map_location=device
))


detector.to(device)
detector.eval()

# -----------------------------
# LOAD SR MODEL
# -----------------------------
sr_model = SRNOInspired().to(device)

sr_model.load_state_dict(torch.load(
    "C:/Users/Mardyson Justin/Thesis/SRNO_Checkpoints5/180k_30e/best_model.pth",
    map_location=device
))


sr_model.eval()

SRNOInspired(
  (encoder): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(64, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
  )
  (mlp): Sequential(
    (0): Linear(in_features=258, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=3, bias=True)
  )
)

In [4]:
def frcnn_detect(image_np, model, device, thresh=0.05):

    img = TF.to_tensor(image_np).to(device)

    with torch.no_grad():
        out = model([img])[0]

    boxes = out["boxes"].detach().cpu().numpy()
    scores = out["scores"].detach().cpu().numpy()
    labels = out["labels"].detach().cpu().numpy()

    keep_boxes = []
    keep_scores = []
    keep_labels = []

    for b, s, l in zip(boxes, scores, labels):
        if s < thresh:
            continue
        if l == 0:
            continue

        keep_boxes.append(b)
        keep_scores.append(float(s))
        keep_labels.append(int(l - 1))

    return keep_boxes, keep_scores, keep_labels

In [5]:
from torch import autocast
from PIL import Image
import numpy as np
import cv2
import torchvision.transforms.functional as TF
import torch

def detect_sr_icro(
    image_path,
    detector,
    sr_model,
    device,
    tau_low=0.3,
    tau_high=0.5,
    min_alpha=1.5,
    alpha_max=4,
    epsilon=0.09,
    max_iter=6,
    alpha_blend_default=0.5,
    max_roi_size=128,
    save_output=False,
    output_path=None
):

    def size_category(w, h):
        area = w * h
        if area < 1024:
            return "tiny"
        elif area < 9216:
            return "small"
        else:
            return "medium"

    image = Image.open(image_path).convert("RGB")
    current_img = np.array(image)
    metrics_log = []

    # ✅ FIX tracking
    num_objects_estimate = 1000
    object_scales = [None] * num_objects_estimate
    object_iterations = [None] * num_objects_estimate

    for iteration in range(max_iter):

        boxes, confs, labels = frcnn_detect(current_img, detector, device)

        if boxes is None or len(boxes) == 0:
            break

        avg_conf_before = np.mean(confs)

        updated_regions = 0
        scales_used = []

        candidate_updates = []
        original_img_copy = current_img.copy()

        for idx, box in enumerate(boxes):

            if idx > 50:
                break

            x1, y1, x2, y2 = map(int, box)
            conf = float(confs[idx])

            x1 = max(0, x1)
            y1 = max(0, y1)
            x2 = min(current_img.shape[1], x2)
            y2 = min(current_img.shape[0], y2)

            if x2 <= x1 or y2 <= y1:
                continue

            roi = current_img[y1:y2, x1:x2]
            H, W = roi.shape[:2]

            if roi.size == 0 or H > max_roi_size or W > max_roi_size:
                continue

            category = size_category(W, H)

            if conf <= tau_low:
                base_alpha = alpha_max
            else:
                ratio = float((tau_high - conf) / (tau_high - tau_low + 1e-8))
                ratio = np.clip(ratio, 0.0, 1.0)
                base_alpha = min_alpha + (alpha_max - min_alpha) * (ratio ** 1.5)

            size_boost = 1.0 if category == "tiny" else 0.75 if category == "small" else 0.5
            iteration_boost = 1.0 + 0.1 * iteration
            alpha = base_alpha * size_boost * iteration_boost

            # 🔒 HARD SAFETY: force real float (prevents complex propagation)
            alpha = float(np.real(alpha))

            # clamp safely
            alpha = min(alpha, float(alpha_max))
            alpha = max(alpha, 1.0)

            scales_used.append(alpha)

            # ✅ FIX
            object_scales[idx] = alpha
            object_iterations[idx] = iteration + 1

            roi_pil = Image.fromarray(roi)
            lr_tensor = TF.to_tensor(roi_pil).unsqueeze(0).to(device)

            target_h = int(H * alpha)
            target_w = int(W * alpha)

            with torch.no_grad():
                sr = sr_model(lr_tensor, target_h, target_w)

            sr_np = np.array(TF.to_pil_image(sr.squeeze(0).clamp(0, 1).cpu()))
            sr_np = cv2.resize(sr_np, (W, H), interpolation=cv2.INTER_CUBIC)

            alpha_blend_roi = np.clip(alpha_blend_default, 0.2, 0.5)

            blended_roi = (
                alpha_blend_roi * sr_np +
                (1 - alpha_blend_roi) * roi
            ).astype(np.uint8)

            candidate_updates.append((idx, x1, y1, x2, y2, blended_roi, conf))

        for (_, x1, y1, x2, y2, blended_roi, _) in candidate_updates:
            current_img[y1:y2, x1:x2] = blended_roi

        boxes_after, confs_after, _ = frcnn_detect(current_img, detector, device)

        if boxes_after is None or len(boxes_after) == 0:
            break

        avg_conf_after = np.mean(confs_after)

        for (obj_idx, x1, y1, x2, y2, _, old_conf) in candidate_updates:

            new_conf = confs_after[obj_idx] if obj_idx < len(confs_after) else old_conf

            if new_conf < old_conf:
                current_img[y1:y2, x1:x2] = original_img_copy[y1:y2, x1:x2]

        delta_conf = avg_conf_after - avg_conf_before

        metrics_log.append({
            "iteration": iteration + 1,
            "avg_conf_before": float(avg_conf_before),
            "avg_conf_after": float(avg_conf_after),
            "delta_conf": float(delta_conf),
            "updated_regions": updated_regions
        })

        print(f"\nIteration {iteration+1}")
        print(f"Delta Confidence: {delta_conf:.4f}")

        if abs(delta_conf) < epsilon:
            break

    valid_len = len(boxes) if boxes is not None else 0

    return current_img, metrics_log, object_scales[:valid_len], object_iterations[:valid_len]

In [6]:
def detect_sr_icro_fixed(
    image_path,
    detector,
    sr_model,
    device,
    tau_low=0.3,
    tau_high=0.5,
    fixed_scales=[2, 3, 4],
    epsilon=0.05,
    max_iter=6,
    alpha_blend_default=0.5,
    max_roi_size=128,
    save_output=False,
    output_path=None
):

    def size_category(w, h):
        area = w * h
        if area < 1024:
            return "tiny"
        elif area < 9216:
            return "small"
        else:
            return "medium"

    image = Image.open(image_path).convert("RGB")
    current_img = np.array(image)
    metrics_log = []

    # ✅ FIX: tracking arrays
    num_objects_estimate = 1000
    object_scales = [None] * num_objects_estimate
    object_iterations = [None] * num_objects_estimate

    for iteration in range(max_iter):

        boxes, confs, labels = frcnn_detect(current_img, detector, device)

        if boxes is None or len(boxes) == 0:
            break

        avg_conf_before = np.mean(confs)

        updated_regions = 0
        scales_used = []

        candidate_updates = []
        original_img_copy = current_img.copy()

        for idx, box in enumerate(boxes):

            if idx > 50:
                break

            x1, y1, x2, y2 = map(int, box)
            conf = float(confs[idx])

            x1 = max(0, x1)
            y1 = max(0, y1)
            x2 = min(current_img.shape[1], x2)
            y2 = min(current_img.shape[0], y2)

            if x2 <= x1 or y2 <= y1:
                continue

            roi = current_img[y1:y2, x1:x2]
            H, W = roi.shape[:2]

            if roi.size == 0 or H > max_roi_size or W > max_roi_size:
                continue

            category = size_category(W, H)

            if conf >= tau_high:
                alpha = 1.0
            else:
                if category == "tiny":
                    alpha = fixed_scales[min(iteration, len(fixed_scales) - 1)]
                elif category == "small":
                    alpha = fixed_scales[min(iteration, 1)]
                else:
                    alpha = fixed_scales[0]

            scales_used.append(alpha)

            # ✅ FIX: store per object
            object_scales[idx] = alpha
            object_iterations[idx] = iteration + 1

            roi_pil = Image.fromarray(roi)
            lr_tensor = TF.to_tensor(roi_pil).unsqueeze(0).to(device)

            target_h = int(H * alpha)
            target_w = int(W * alpha)

            with torch.no_grad():
                sr = sr_model(lr_tensor, target_h, target_w)

            sr_np = np.array(TF.to_pil_image(sr.squeeze(0).clamp(0, 1).cpu()))
            sr_np = cv2.resize(sr_np, (W, H), interpolation=cv2.INTER_CUBIC)

            alpha_blend_roi = np.clip(alpha_blend_default, 0.4, 0.7)

            blended_roi = (
                alpha_blend_roi * sr_np +
                (1 - alpha_blend_roi) * roi
            ).astype(np.uint8)

            candidate_updates.append((idx, x1, y1, x2, y2, blended_roi, conf))

        for (_, x1, y1, x2, y2, blended_roi, _) in candidate_updates:
            current_img[y1:y2, x1:x2] = blended_roi

        boxes_after, confs_after, _ = frcnn_detect(current_img, detector, device)

        if boxes_after is None or len(boxes_after) == 0:
            break

        avg_conf_after = np.mean(confs_after)

        for (obj_idx, x1, y1, x2, y2, _, old_conf) in candidate_updates:

            new_conf = confs_after[obj_idx] if obj_idx < len(confs_after) else old_conf

            if new_conf < old_conf:
                current_img[y1:y2, x1:x2] = original_img_copy[y1:y2, x1:x2]

        delta_conf = avg_conf_after - avg_conf_before

        # ✅ FIX: log metrics
        metrics_log.append({
            "iteration": iteration + 1,
            "avg_conf_before": float(avg_conf_before),
            "avg_conf_after": float(avg_conf_after),
            "delta_conf": float(delta_conf),
            "updated_regions": updated_regions
        })

        print(f"\nIteration {iteration+1}")
        print(f"Avg Scaling Factor   : {np.mean(scales_used) if scales_used else 1.0:.2f}")
        print(f"Updated Regions      : {updated_regions}")
        print(f"Delta Confidence     : {delta_conf:.4f}")

        if abs(delta_conf) < epsilon:
            break

    valid_len = len(boxes) if boxes is not None else 0

    return current_img, metrics_log, object_scales[:valid_len], object_iterations[:valid_len]

In [6]:
from ultralytics import YOLO

# 🔥 LOAD YOUR TRAINED YOLO MODEL
yolo_model = YOLO("C:/Users/Mardyson Justin/Thesis/yolov9c_visdrone_finetune15/weights/best.pt")

# set device (CPU or GPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
yolo_model.to(device)

YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(128, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): RepNCSPELAN4(
        (cv1): Conv(
          (conv): Conv2d(128, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(128, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Sequential(
          (0): RepCSP(
            (cv1): Conv(
              (conv): Conv2d(64, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
              

In [ ]:
#yolo icro
from torch import autocast
from PIL import Image
import numpy as np
import cv2
import torchvision.transforms.functional as TF
import torch

def detect_sr_icro_yolo(
    image_path,
    tau_low=0.3,
    tau_high=0.5,
    min_alpha=1.5,
    alpha_max=4,
    epsilon=0.05,
    max_iter=6,
    alpha_blend_default=0.5,
    max_roi_size=128,
    save_output=False,
    output_path=None
):

    def size_category(w, h):
        area = w*h
        if area < 1024:
            return "tiny"
        elif area < 9216:
            return "small"
        else:
            return "medium"

    # Alpha caps per category
    alpha_caps = {"tiny": 3.0, "small": 2.0, "medium": 1.8}

    image = Image.open(image_path).convert("RGB")
    current_img = np.array(image)
    metrics_log = []

    num_objects_estimate = 1000
    object_scales = [1.0] * num_objects_estimate
    object_iterations = [1] * num_objects_estimate

    for iteration in range(max_iter):
        results = yolo_model(current_img, verbose=False)[0]
        boxes = results.boxes

        if boxes is None or len(boxes) == 0:
            print("No detections.")
            break

        confs_before = boxes.conf.cpu().numpy()
        avg_conf_before = np.mean(confs_before)
        updated_regions = 0
        scales_used = []

        tiny_objects_present = False

        for idx, box in enumerate(boxes):
            conf = float(box.conf)
            x1, y1, x2, y2 = map(int, box.xyxy.cpu().numpy()[0])
            roi = current_img[y1:y2, x1:x2]

            H, W = roi.shape[:2] if roi.size > 0 else (0, 0)
            category = size_category(W, H)

            # Skip SR if already high confidence
            if conf >= tau_high or roi.size == 0 or H > max_roi_size or W > max_roi_size:
                object_scales[idx] = max(object_scales[idx], min_alpha)
                object_iterations[idx] = max(object_iterations[idx], iteration + 1)
                continue

            if category == "tiny":
                tiny_objects_present = True
                max_obj_iter = max_iter
            elif category == "small":
                max_obj_iter = max_iter - 1
            else:
                max_obj_iter = 1

            if iteration + 1 > max_obj_iter:
                object_scales[idx] = max(object_scales[idx], min_alpha)
                object_iterations[idx] = max(object_iterations[idx], iteration + 1)
                continue

            # Adaptive alpha
            uncertainty = (tau_high - conf) / (tau_high - tau_low)
            uncertainty = np.clip(uncertainty, 0, 1)
            uncertainty = 0.7 * uncertainty + 0.3 * 0.5

            base_alpha = min_alpha + (alpha_max - min_alpha) * (uncertainty ** 1.2)

            # size-aware damping (IMPORTANT for LPIPS)
            if category == "tiny":
                size_factor = 0.9
            elif category == "small":
                size_factor = 0.75
            else:
                size_factor = 0.6

            # iteration decay (NOT boost)
            iteration_factor = 1.0 / (1.0 + 0.15 * iteration)

            alpha = base_alpha * size_factor * iteration_factor

            # final perceptual clamp (VERY IMPORTANT)
            alpha = np.clip(alpha, 1.1, 2.6)

            # Cap alpha per category
            alpha = min(alpha, alpha_caps[category])
            scales_used.append(alpha)

            # ROI tensor
            roi_pil = Image.fromarray(roi)
            model_device = next(sr_model.parameters()).device
            lr_tensor = TF.to_tensor(roi_pil).unsqueeze(0).to(model_device)
            if next(sr_model.parameters()).dtype == torch.float16:
                lr_tensor = lr_tensor.half()

            target_h = int(H * alpha)
            target_w = int(W * alpha)

            with torch.no_grad(), autocast(device_type='cuda', enabled=(lr_tensor.dtype == torch.float16)):
                torch.cuda.empty_cache()
                sr = sr_model(lr_tensor, target_h, target_w)

            sr_img = TF.to_pil_image(sr.squeeze(0).clamp(0, 1).cpu())
            sr_np = np.array(sr_img)
            sr_np_resized = cv2.resize(sr_np, (W, H), interpolation=cv2.INTER_CUBIC)

            # Confidence-weighted blending with safe max
            delta_conf_roi = tau_high - conf
            alpha_blend_roi = alpha_blend_default * (0.65 + 0.2 * (1 - uncertainty))
            alpha_blend_roi = np.clip(alpha_blend_roi, 0.35, 0.65)

            blended_roi = (alpha_blend_roi * sr_np_resized + (1 - alpha_blend_roi) * roi).astype(np.uint8)
            current_img[y1:y2, x1:x2] = blended_roi
            updated_regions += 1

            # Update tracking
            object_scales[idx] = max(object_scales[idx], alpha)
            object_iterations[idx] = max(object_iterations[idx], iteration + 1)

        # Post-iteration metrics
        results_after = yolo_model(current_img, verbose=False)[0]
        boxes_after = results_after.boxes
        if boxes_after is None or len(boxes_after) == 0:
            break

        confs_after = boxes_after.conf.cpu().numpy()
        avg_conf_after = np.mean(confs_after)
        delta_conf = avg_conf_after - avg_conf_before
        avg_scale = np.mean(scales_used) if scales_used else 1.0

        metrics_log.append({
            "iteration": iteration + 1,
            "avg_scaling_factor": float(avg_scale),
            "regions_updated": updated_regions,
            "avg_conf_before": float(avg_conf_before),
            "avg_conf_after": float(avg_conf_after),
            "delta_conf": float(delta_conf)
        })

        print(f"\nIteration {iteration+1}")
        print(f"Avg Scaling Factor   : {avg_scale:.2f}")
        print(f"Updated Regions      : {updated_regions}")
        print(f"Avg Confidence Before: {avg_conf_before:.4f}")
        print(f"Avg Confidence After : {avg_conf_after:.4f}")
        print(f"Delta Confidence     : {delta_conf:.4f}")

        # Convergence checks
        delta_conf_effective = delta_conf * 0.5 if tiny_objects_present else delta_conf
        if abs(delta_conf_effective) < epsilon and not tiny_objects_present:
            print("Converged.")
            break
        if avg_conf_after >= tau_high:
            print("Reached Stable High Confidence.")
            break

    if save_output and output_path is not None:
        cv2.imwrite(output_path, current_img)

    return current_img, metrics_log, object_scales[:len(boxes)], object_iterations[:len(boxes)]

In [8]:
#yolo icro fixed
def detect_sr_icro_fixed_yolo(
    image_path,
    tau_low=0.3,
    tau_high=0.5,
    fixed_scales=[2, 3, 4],
    epsilon=0.05,
    max_iter=6,
    alpha_blend_default=0.5,
    max_roi_size=128,
    save_output=False,
    output_path=None
):

    def size_category(w, h):
        area = w*h
        if area < 1024:
            return "tiny"
        elif area < 9216:
            return "small"
        else:
            return "medium"

    image = Image.open(image_path).convert("RGB")
    current_img = np.array(image)
    metrics_log = []

    num_objects_estimate = 1000
    object_scales = [1.0] * num_objects_estimate
    object_iterations = [1] * num_objects_estimate

    for iteration in range(max_iter):
        results = yolo_model(current_img, verbose=False)[0]
        boxes = results.boxes

        if boxes is None or len(boxes) == 0:
            print("No detections.")
            break

        confs_before = boxes.conf.cpu().numpy()
        avg_conf_before = np.mean(confs_before)
        updated_regions = 0
        scales_used = []

        for idx, box in enumerate(boxes):
            conf = float(box.conf)
            x1, y1, x2, y2 = map(int, box.xyxy.cpu().numpy()[0])
            roi = current_img[y1:y2, x1:x2]

            H, W = roi.shape[:2] if roi.size > 0 else (0,0)
            category = size_category(W, H)

            if roi.size == 0 or H > max_roi_size or W > max_roi_size:
                continue

            # ------------------------
            # Dynamic scale selection
            # ------------------------
            if conf >= tau_high:
                alpha = 1.0
            else:
                if category == "tiny":
                    alpha = fixed_scales[min(iteration, len(fixed_scales)-1)]
                elif category == "small":
                    alpha = fixed_scales[min(iteration, 1)]
                else:
                    alpha = fixed_scales[0]

            scales_used.append(alpha)

            # ROI tensor
            roi_pil = Image.fromarray(roi)
            model_device = next(sr_model.parameters()).device
            lr_tensor = TF.to_tensor(roi_pil).unsqueeze(0).to(model_device)

            if next(sr_model.parameters()).dtype == torch.float16:
                lr_tensor = lr_tensor.half()

            target_h = int(H * alpha)
            target_w = int(W * alpha)

            with torch.no_grad(), autocast(device_type='cuda', enabled=(lr_tensor.dtype == torch.float16)):
                torch.cuda.empty_cache()
                sr = sr_model(lr_tensor, target_h, target_w)

            sr_img = TF.to_pil_image(sr.squeeze(0).clamp(0, 1).cpu())
            sr_np = np.array(sr_img)

            # keep cubic
            sr_np_resized = cv2.resize(sr_np, (W, H), interpolation=cv2.INTER_CUBIC)

            # ✅ FIXED: stronger lower bound to prevent artifacts
            delta_conf_roi = max(tau_high - conf, 0)
            alpha_blend_roi = alpha_blend_default * (delta_conf_roi / (tau_high - tau_low))
            alpha_blend_roi = np.clip(alpha_blend_roi, 0.4, 0.7)

            blended_roi = (alpha_blend_roi * sr_np_resized + (1 - alpha_blend_roi) * roi).astype(np.uint8)
            current_img[y1:y2, x1:x2] = blended_roi
            updated_regions += 1

            object_scales[idx] = max(object_scales[idx], alpha)
            object_iterations[idx] = max(object_iterations[idx], iteration + 1)

        # ------------------------
        # Confidence check
        # ------------------------
        results_after = yolo_model(current_img, verbose=False)[0]
        boxes_after = results_after.boxes
        if boxes_after is None or len(boxes_after) == 0:
            break

        confs_after = boxes_after.conf.cpu().numpy()
        avg_conf_after = np.mean(confs_after)
        delta_conf = avg_conf_after - avg_conf_before
        avg_scale = np.mean(scales_used) if scales_used else 1.0

        metrics_log.append({
            "iteration": iteration + 1,
            "avg_scaling_factor": float(avg_scale),
            "regions_updated": updated_regions,
            "avg_conf_before": float(avg_conf_before),
            "avg_conf_after": float(avg_conf_after),
            "delta_conf": float(delta_conf)
        })

        print(f"\nIteration {iteration+1}")
        print(f"Avg Scaling Factor   : {avg_scale:.2f}")
        print(f"Updated Regions      : {updated_regions}")
        print(f"Avg Confidence Before: {avg_conf_before:.4f}")
        print(f"Avg Confidence After : {avg_conf_after:.4f}")
        print(f"Delta Confidence     : {delta_conf:.4f}")

        if abs(delta_conf) < epsilon:
            print("Converged.")
            break
        if avg_conf_after >= tau_high:
            print("Reached Stable High Confidence.")
            break

    if save_output and output_path is not None:
        cv2.imwrite(output_path, current_img)

    return current_img, metrics_log, object_scales[:len(boxes)], object_iterations[:len(boxes)]

In [ ]:
# -----------------------------
# VRAM-SAFE VALIDATION (OPTION A: CPU MODE)
# -----------------------------
import os
import time
import cv2
import torch
import pandas as pd
import numpy as np
from PIL import Image
import torchvision.transforms.functional as TF
from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from thop import profile
import lpips
import gc

#withflopss

NUM_CLASSES = 11

# -----------------------------
# DEVICE FOR DETECTOR = CPU (IMPORTANT FIX)
# -----------------------------
device = "cpu"

# -----------------------------
# DETECTOR (CPU ONLY)
# -----------------------------
detector = fasterrcnn_mobilenet_v3_large_fpn(weights=None)
in_features = detector.roi_heads.box_predictor.cls_score.in_features
detector.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)

detector.load_state_dict(
    torch.load(
        "C:/Users/Mardyson Justin/Thesis/FCNN3/checkpoints/best_model.pth",
        map_location="cpu"
    ),
    strict=True
)

detector.to(device)
detector.eval()

# -----------------------------
# SR MODEL (CPU ONLY - IMPORTANT FIX)
# -----------------------------
sr_model = SRNOInspired()
sr_model.load_state_dict(
    torch.load(
        "C:/Users/Mardyson Justin/Thesis/SRNO_Checkpoints5/180k_30e/best_model.pth",
        map_location="cpu"
    )
)
sr_model.to("cpu")
sr_model.eval()

# -----------------------------
# LPIPS (CPU)
# -----------------------------
lpips_model = lpips.LPIPS(net='alex').cpu()
lpips_model.eval()

def compute_flops(model, input_tensor, extra_args=None):
    model.eval()
    with torch.no_grad():
        if extra_args is not None:
            flops, params = profile(model, inputs=extra_args, verbose=False)
        else:
            flops, params = profile(model, inputs=(input_tensor,), verbose=False)
    return flops, params


dummy_img = TF.to_tensor(np.zeros((720, 1280, 3), dtype=np.uint8)).unsqueeze(0)

flops, params = profile(detector, inputs=( [dummy_img.squeeze(0)], ), verbose=False)

det_flops = flops
det_params = params
print("Detector FLOPs:", det_flops)

# -----------------------------
# SR MODEL FLOPs (RUN ONCE)
# -----------------------------
dummy_lr = torch.randn(1, 3, 128, 128)
sr_flops, sr_params = compute_flops(
    sr_model,
    dummy_lr,
    extra_args=(dummy_lr, 256, 256)
)


# -----------------------------
# CONSTANTS
# -----------------------------
VALID_CLASSES = [3, 5, 8]

UAVDT_TO_VISDRONE = {0: 3, 1: 5, 2: 8}
VISDRONE_TO_CUSTOM = {3: 0, 5: 1, 8: 2}


# -----------------------------
# DETECTOR FLOPs (RUN ONCE)
# -----------------------------
try:
    sr_flops, sr_params = compute_flops(sr_model, dummy_lr, extra_args=(dummy_lr, 256, 256))
except Exception as e:
    print("SR FLOPs failed:", e)
    sr_flops, sr_params = 0, 0
# -----------------------------
# HELPERS
# -----------------------------
def compute_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    areaA = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    areaB = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    union = areaA + areaB - inter
    return 0 if union == 0 else inter / union


def get_matching_conf(box, results):
    if results.boxes is None:
        return 0

    boxes = results.boxes.xyxy.cpu().numpy()
    confs = results.boxes.conf.cpu().numpy()

    best_score = 0
    x1, y1, x2, y2 = box
    cx = (x1 + x2) / 2
    cy = (y1 + y2) / 2

    for b, c in zip(boxes, confs):
        bx1, by1, bx2, by2 = b

        iou = compute_iou(box, b)

        bx = (bx1 + bx2) / 2
        by = (by1 + by2) / 2
        dist = np.sqrt((cx - bx)**2 + (cy - by)**2)

        # combine both signals instead of hard thresholds
        score = (0.7 * iou + 0.3 * (1 / (1 + dist / 100))) * c

        if score > best_score:
            best_score = score

    return float(best_score)


def get_matching_class(box, results):
    if results.boxes is None:
        return -1

    boxes = results.boxes.xyxy.cpu().numpy()
    classes = results.boxes.cls.cpu().numpy()

    best_iou = 0
    best_cls = -1
    best_dist = 1e9
    best_center_cls = -1

    x1, y1, x2, y2 = box
    cx = (x1 + x2) / 2
    cy = (y1 + y2) / 2

    for b, c in zip(boxes, classes):
        bx1, by1, bx2, by2 = b
        iou = compute_iou(box, b)

        if iou > best_iou:
            best_iou = iou
            best_cls = c

        bx = (bx1 + bx2) / 2
        by = (by1 + by2) / 2
        dist = np.sqrt((cx - bx) ** 2 + (cy - by) ** 2)

        if dist < best_dist:
            best_dist = dist
            best_center_cls = c

    if best_iou > 0.05:
        return int(best_cls)
    if best_dist < 50:
        return int(best_center_cls)
    return -1


def get_gt_class(box, gt_boxes, gt_classes):
    best_iou = 0
    best_cls = -1
    for b, c in zip(gt_boxes, gt_classes):
        iou = compute_iou(box, b)
        if iou > best_iou:
            best_iou = iou
            best_cls = c
    return int(best_cls) if best_iou > 0.5 else -1


def size_category(w, h):
    area = w * h
    if area < 1024:
        return "tiny"
    elif area < 9216:
        return "small"
    return "medium"


def compute_object_metrics(sr_img, hr_img, box):
    x1, y1, x2, y2 = map(int, box)

    H, W = hr_img.shape[:2]

    # clamp
    x1 = max(0, min(x1, W - 1))
    x2 = max(0, min(x2, W))
    y1 = max(0, min(y1, H - 1))
    y2 = max(0, min(y2, H))

    # fix invalid boxes
    if x2 <= x1 + 1 or y2 <= y1 + 1:
        return 0.0, 0.0, 0.0

    crop_sr = sr_img[y1:y2, x1:x2]
    crop_hr = hr_img[y1:y2, x1:x2]

    # resize safety fallback
    if crop_sr.shape[0] < 8 or crop_sr.shape[1] < 8:
        crop_sr = cv2.resize(crop_sr, (16, 16))
        crop_hr = cv2.resize(crop_hr, (16, 16))

    try:
        psnr = peak_signal_noise_ratio(crop_hr, crop_sr, data_range=255)
    except:
        psnr = 0.0

    try:
        ssim = structural_similarity(
            crop_hr, crop_sr,
            channel_axis=2,
            data_range=255,
            win_size=min(7, crop_hr.shape[0] - 1 if crop_hr.shape[0] % 2 == 0 else crop_hr.shape[0])
        )
    except:
        ssim = 0.0

    try:
        crop_sr_lp = cv2.resize(crop_sr, (32, 32))
        crop_hr_lp = cv2.resize(crop_hr, (32, 32))

        t1 = TF.to_tensor(crop_sr_lp).unsqueeze(0).float() * 2 - 1
        t2 = TF.to_tensor(crop_hr_lp).unsqueeze(0).float() * 2 - 1

        with torch.no_grad():
            lp = lpips_model(t1, t2).item()

    except:
        lp = 0.0

    return psnr, ssim, lp



# -----------------------------
# RUN DETECTOR (CPU SAFE)
# -----------------------------
def run_detector(img):
    img_tensor = TF.to_tensor(img).cpu()

    with torch.no_grad():
        outputs = detector([img_tensor])[0]

    boxes = outputs['boxes'].cpu().numpy()
    scores = outputs['scores'].cpu().numpy()
    labels = outputs['labels'].cpu().numpy()

    filtered_boxes = []
    filtered_scores = []
    filtered_labels = []

    for b, s, l in zip(boxes, scores, labels):
        if s > 0.05:
            filtered_boxes.append(b)
            filtered_scores.append(s)
            filtered_labels.append(l - 1)

    class Result:
        pass

    r = Result()
    r.boxes = None

    if len(filtered_boxes) > 0:
        class Boxes:
            pass
        r.boxes = Boxes()
        r.boxes.xyxy = torch.tensor(filtered_boxes)
        r.boxes.conf = torch.tensor(filtered_scores)
        r.boxes.cls = torch.tensor(filtered_labels)

    return r

# -----------------------------
# MAIN LOOP SETUP
# -----------------------------
test_folder = "C:/Users/Mardyson Justin/Thesis/UAVDT/test/images/"
gt_folder = "C:/Users/Mardyson Justin/Thesis/UAVDT/labels/"
excel_path = "C:/Users/Mardyson Justin/Thesis/FCNNVal/FCNNTest_UAVDT01_Finalized_results.xlsx"

os.makedirs(os.path.dirname(excel_path), exist_ok=True)

rows = []

all_images = sorted([
    f for f in os.listdir(test_folder)
    if f.lower().endswith((".jpg", ".png", ".jpeg"))
])

# -----------------------------
# LOOP
# -----------------------------
for img_name in all_images:
    print("Processing:", img_name)

    img_path = os.path.join(test_folder, img_name)

    hr_img = Image.open(img_path).convert("RGB")
    hr_np = np.array(hr_img)
    H, W = hr_np.shape[:2]

    # -----------------------------
    # GT
    # -----------------------------
    gt_path = os.path.join(gt_folder, img_name.replace(".jpg", ".txt"))
    gt_boxes, gt_classes = [], []

    if os.path.exists(gt_path):
        with open(gt_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue

                cls = int(parts[0])
                xc, yc, w, h = map(float, parts[1:5])

                x1 = (xc - w / 2) * W
                y1 = (yc - h / 2) * H
                x2 = (xc + w / 2) * W
                y2 = (yc + h / 2) * H

                if cls in UAVDT_TO_VISDRONE:
                    vis = UAVDT_TO_VISDRONE[cls]
                    if vis in VALID_CLASSES:
                        gt_boxes.append([x1, y1, x2, y2])
                        gt_classes.append(VISDRONE_TO_CUSTOM[vis])

    # -----------------------------
    # LR
    # -----------------------------
    lr = cv2.resize(hr_np, (int(W / 3), int(H / 3)))

    # -----------------------------
    # DETECTION (CPU)
    # -----------------------------
    start = time.perf_counter()
    yolo_results_cpu = run_detector(hr_np)
    yolo_results_lr = run_detector(lr)
    scale_x = W / lr.shape[1]
    scale_y = H / lr.shape[0]

    if yolo_results_lr.boxes is not None:
        boxes = yolo_results_lr.boxes.xyxy.numpy()
        boxes[:, [0, 2]] *= scale_x
        boxes[:, [1, 3]] *= scale_y
        yolo_results_lr.boxes.xyxy = torch.tensor(boxes)
    yolo_results_hr = run_detector(hr_np)
    runtime_frcnn = time.perf_counter() - start

    if yolo_results_cpu.boxes is None:
        continue

    base_boxes = yolo_results_cpu.boxes.xyxy.numpy()
    base_classes = yolo_results_cpu.boxes.cls.numpy()
    num_objects = max(1, len(base_boxes))

    # -----------------------------
    # SR (CPU ONLY)
    # -----------------------------
    lr_tensor = TF.to_tensor(lr).unsqueeze(0).cpu()

    start = time.perf_counter()
    with torch.inference_mode():
        sr = sr_model(lr_tensor, H, W)

    sr = sr.detach().cpu()
    runtime_sr_img = time.perf_counter() - start

    sr_img = sr.squeeze().clamp(0, 1)
    sr_np = (sr_img.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    sr_results = run_detector(sr_np)
    print("SR boxes:", None if sr_results.boxes is None else len(sr_results.boxes.xyxy))

    # -----------------------------
    # ICRO (UNCHANGED)
    # -----------------------------
    start = time.perf_counter()
    sr_fixed, *_ = detect_sr_icro_fixed(img_path, detector, sr_model, device, save_output=False)
    runtime_icro_fixed_img = time.perf_counter() - start

    start = time.perf_counter()
    sr_adaptive, *_ = detect_sr_icro(img_path, detector, sr_model, device, save_output=False)
    runtime_icro_adapt_img = time.perf_counter() - start

    sr_fixed_results = run_detector(sr_fixed)
    sr_adaptive_results = run_detector(sr_adaptive)

    # -----------------------------
    # OBJECT LOOP
    # -----------------------------
    total_area = H * W

    for i, (box, cls) in enumerate(zip(base_boxes, base_classes)):
        conf_frcnn = get_matching_conf(box, yolo_results_lr)
            
        x1, y1, x2, y2 = map(int, box)
        w, h = x2 - x1, y2 - y1
        if w < 4 or h < 4:
            continue

        size = size_category(w, h)
        gt_class = get_gt_class(box, gt_boxes, gt_classes)

        roi_area = w * h
        area_ratio = roi_area / total_area
        runtime_per_obj = runtime_frcnn / num_objects

        runtime_sr = (runtime_per_obj + runtime_sr_img * area_ratio) / num_objects
        runtime_icro_fixed = (runtime_per_obj + runtime_icro_fixed_img * area_ratio) / num_objects
        runtime_icro_adaptive = (runtime_per_obj + runtime_icro_adapt_img * area_ratio) / num_objects

        conf_sr = get_matching_conf(box, sr_results)
        conf_fixed = get_matching_conf(box, sr_fixed_results)
        conf_adaptive = get_matching_conf(box, sr_adaptive_results)

        psnr_sr, ssim_sr, lpips_sr = compute_object_metrics(sr_np, hr_np, box)
        psnr_fixed, ssim_fixed, lpips_fixed = compute_object_metrics(sr_fixed, hr_np, box)
        psnr_adaptive, ssim_adaptive, lpips_adaptive = compute_object_metrics(sr_adaptive, hr_np, box)

        rows.append({
            "image_name": img_name,
            "object_id": i,
            "size": size,
            "gt_class": gt_class,

            "conf_frcnn": conf_frcnn,
            "conf_sr": conf_sr,
            "conf_icro_fixed": conf_fixed,
            "conf_icro_adaptive": conf_adaptive,

            "runtime_sr": runtime_sr,
            "runtime_icro_fixed": runtime_icro_fixed,
            "runtime_icro_adaptive": runtime_icro_adaptive,

            "psnr_sr": psnr_sr,
            "ssim_sr": ssim_sr,
            "lpips_sr": lpips_sr,

            "psnr_icro_fixed": psnr_fixed,
            "ssim_icro_fixed": ssim_fixed,
            "lpips_icro_fixed": lpips_fixed,

            "psnr_icro_adaptive": psnr_adaptive,
            "ssim_icro_adaptive": ssim_adaptive,
            "lpips_icro_adaptive": lpips_adaptive
        })

    # -----------------------------
    # CLEANUP (SAFE CPU MODE)
    # -----------------------------
    del yolo_results_cpu, sr_np, sr_fixed, sr_adaptive
    gc.collect()

# -----------------------------
# SAVE
# -----------------------------
df = pd.DataFrame(rows)
df.to_excel(excel_path, index=False)

print("Validation complete") # this for frcnn orig

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: c:\Users\Mardyson Justin\Thesis\venv\Lib\site-packages\lpips\weights\v0.1\alex.pth
Detector FLOPs: 7246112088.0
Processing: DJI-405-720p00331.jpg
SR boxes: 6

Iteration 1
Avg Scaling Factor   : 1.00
Updated Regions      : 0
Delta Confidence     : 0.0283

Iteration 1
Delta Confidence: 0.0291
Processing: DJI-405-720p00351.jpg
SR boxes: 14

Iteration 1
Avg Scaling Factor   : 1.23
Updated Regions      : 0
Delta Confidence     : 0.0961

Iteration 2
Avg Scaling Factor   : 1.15
Updated Regions      : 0
Delta Confidence     : -0.0138

Iteration 1
Delta Confidence: 0.0366
Processing: DJI-405-720p02001.jpg
SR boxes: 19

Iteration 1
Avg Scaling Factor   : 1.23
Updated Regions      : 0
Delta Confidence     : 0.1110

Iteration 2
Avg Scaling Factor   : 1.17
Updated Regions      : 0
Delta Confidence     : 0.0833

Iteration 3
Avg Scaling Factor   : 1.00
Updated Regions      : 0
Delta Confidence     : 0.0013

It

In [16]:
# -----------------------------
# VRAM-SAFE VALIDATION (OPTION A: CPU MODE)
# -----------------------------
import os
import time
import cv2
import torch
import pandas as pd
import numpy as np
from PIL import Image
import torchvision.transforms.functional as TF
from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
import lpips
import gc

#withflopss

NUM_CLASSES = 11

# -----------------------------
# DEVICE FOR DETECTOR = CPU (IMPORTANT FIX)
# -----------------------------
device = "cpu"


# -----------------------------
# DETECTOR (CPU ONLY)
# -----------------------------
detector = fasterrcnn_mobilenet_v3_large_fpn(weights=None)
in_features = detector.roi_heads.box_predictor.cls_score.in_features
detector.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)

detector.load_state_dict(
    torch.load(
        "C:/Users/Mardyson Justin/Thesis/FCNN3/checkpoints/best_model.pth",
        map_location="cpu"
    ),
    strict=True
)

detector.to(device)
detector.eval()

# -----------------------------
# SR MODEL (CPU ONLY - IMPORTANT FIX)
# -----------------------------
sr_model = SRNOInspired()
sr_model.load_state_dict(
    torch.load(
        "C:/Users/Mardyson Justin/Thesis/SRNO_Checkpoints5/180k_30e/best_model.pth",
        map_location="cpu"
    )
)
sr_model.to("cpu")
sr_model.eval()

# -----------------------------
# LPIPS (CPU)
# -----------------------------
lpips_model = lpips.LPIPS(net='alex').cpu()
lpips_model.eval()


# -----------------------------
# CONSTANTS
# -----------------------------
VALID_CLASSES = [3, 5, 8]

UAVDT_TO_VISDRONE = {0: 3, 1: 5, 2: 8}
VISDRONE_TO_CUSTOM = {3: 0, 5: 1, 8: 2}

# -----------------------------
# HELPERS
# -----------------------------
def compute_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    areaA = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    areaB = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    union = areaA + areaB - inter
    return 0 if union == 0 else inter / union


def get_matching_conf(box, results):
    if results.boxes is None:
        return 0

    boxes = results.boxes.xyxy.cpu().numpy()
    confs = results.boxes.conf.cpu().numpy()

    best_score = 0
    x1, y1, x2, y2 = box
    cx = (x1 + x2) / 2
    cy = (y1 + y2) / 2

    for b, c in zip(boxes, confs):
        bx1, by1, bx2, by2 = b

        iou = compute_iou(box, b)

        bx = (bx1 + bx2) / 2
        by = (by1 + by2) / 2
        dist = np.sqrt((cx - bx)**2 + (cy - by)**2)

        # combine both signals instead of hard thresholds
        score = (0.7 * iou + 0.3 * (1 / (1 + dist / 100))) * c

        if score > best_score:
            best_score = score

    return float(best_score)


def get_matching_class(box, results):
    if results.boxes is None:
        return -1

    boxes = results.boxes.xyxy.cpu().numpy()
    classes = results.boxes.cls.cpu().numpy()

    best_iou = 0
    best_cls = -1
    best_dist = 1e9
    best_center_cls = -1

    x1, y1, x2, y2 = box
    cx = (x1 + x2) / 2
    cy = (y1 + y2) / 2

    for b, c in zip(boxes, classes):
        bx1, by1, bx2, by2 = b
        iou = compute_iou(box, b)

        if iou > best_iou:
            best_iou = iou
            best_cls = c

        bx = (bx1 + bx2) / 2
        by = (by1 + by2) / 2
        dist = np.sqrt((cx - bx) ** 2 + (cy - by) ** 2)

        if dist < best_dist:
            best_dist = dist
            best_center_cls = c

    if best_iou > 0.05:
        return int(best_cls)
    if best_dist < 50:
        return int(best_center_cls)
    return -1


def get_gt_class(box, gt_boxes, gt_classes):
    best_iou = 0
    best_cls = -1
    for b, c in zip(gt_boxes, gt_classes):
        iou = compute_iou(box, b)
        if iou > best_iou:
            best_iou = iou
            best_cls = c
    return int(best_cls) if best_iou > 0.5 else -1


def size_category(w, h):
    area = w * h
    if area < 1024:
        return "tiny"
    elif area < 9216:
        return "small"
    return "medium"


def compute_object_metrics(sr_img, hr_img, box):
    x1, y1, x2, y2 = map(int, box)

    H, W = hr_img.shape[:2]

    # clamp
    x1 = max(0, min(x1, W - 1))
    x2 = max(0, min(x2, W - 1))
    y1 = max(0, min(y1, H - 1))
    y2 = max(0, min(y2, H - 1))

    # FIX: invalid or empty crop guard
    if x2 <= x1 or y2 <= y1:
        return 0.0, 0.0, 0.0

    crop_sr = sr_img[y1:y2, x1:x2]
    crop_hr = hr_img[y1:y2, x1:x2]

    # EXTRA SAFETY: empty array guard
    if crop_sr is None or crop_hr is None or crop_sr.size == 0 or crop_hr.size == 0:
        return 0.0, 0.0, 0.0

    # resize safety fallback
    if crop_sr.shape[0] < 8 or crop_sr.shape[1] < 8:
        crop_sr = cv2.resize(crop_sr, (16, 16))
        crop_hr = cv2.resize(crop_hr, (16, 16))

    try:
        psnr = peak_signal_noise_ratio(crop_hr, crop_sr, data_range=255)
    except:
        psnr = 0.0

    try:
        ssim = structural_similarity(
            crop_hr, crop_sr,
            channel_axis=2,
            data_range=255,
            win_size=min(7, crop_hr.shape[0] - 1 if crop_hr.shape[0] % 2 == 0 else crop_hr.shape[0])
        )
    except:
        ssim = 0.0

    try:
        crop_sr_lp = cv2.resize(crop_sr, (64, 64))
        crop_hr_lp = cv2.resize(crop_hr, (64, 64))

        t1 = TF.to_tensor(crop_sr_lp).unsqueeze(0).float() * 2 - 1
        t2 = TF.to_tensor(crop_hr_lp).unsqueeze(0).float() * 2 - 1

        with torch.no_grad():
            lp = lpips_model(t1, t2).item()

    except:
        lp = 0.0

    return psnr, ssim, lp


def sr_tile_infer(sr_model, lr_img, tile=128, device="cpu"):
    """
    Safe SR inference using tiling to avoid VRAM explosion
    """
    _, H, W = lr_img.shape
    sr_output = torch.zeros((3, H * 2, W * 2))  # adjust upscale if needed

    stride = tile
    with torch.inference_mode():
        for y in range(0, H, stride):
            for x in range(0, W, stride):

                patch = lr_img[:, y:y+tile, x:x+tile].unsqueeze(0).to(device)

                # IMPORTANT: force small compute graph
                out = sr_model(patch, patch.shape[-2]*2, patch.shape[-1]*2)

                out = out.squeeze(0).cpu()

                oy, ox = y*2, x*2
                ph, pw = out.shape[-2:]

                sr_output[:, oy:oy+ph, ox:ox+pw] = out

                del patch, out

    return sr_output

# -----------------------------
# RUN DETECTOR (CPU SAFE)
# -----------------------------
def run_detector(img):
    img_tensor = TF.to_tensor(img).to(device)

    with torch.inference_mode():
        outputs = detector([img_tensor])[0]

    boxes = outputs['boxes'].cpu().numpy()
    scores = outputs['scores'].cpu().numpy()
    labels = outputs['labels'].cpu().numpy()

    filtered_boxes = []
    filtered_scores = []
    filtered_labels = []

    for b, s, l in zip(boxes, scores, labels):
        if s > 0.05:
            filtered_boxes.append(b)
            filtered_scores.append(s)
            filtered_labels.append(l - 1)

    class Result:
        pass

    r = Result()
    r.boxes = None

    if len(filtered_boxes) > 0:
        class Boxes:
            pass
        r.boxes = Boxes()
        r.boxes.xyxy = torch.tensor(filtered_boxes)
        r.boxes.conf = torch.tensor(filtered_scores)
        r.boxes.cls = torch.tensor(filtered_labels)

    del img_tensor, outputs
    gc.collect()

    return r

# -----------------------------
# MAIN LOOP SETUP
# -----------------------------
test_folder = "C:/Users/Mardyson Justin/Thesis/UAVDT/test/images/"
gt_folder = "C:/Users/Mardyson Justin/Thesis/UAVDT/labels/"
excel_path = "C:/Users/Mardyson Justin/Thesis/FCNNVal/FCNNTest_UAVDT01_Finalized_results.xlsx"

os.makedirs(os.path.dirname(excel_path), exist_ok=True)

rows = []

all_images = sorted([
    f for f in os.listdir(test_folder)
    if f.lower().endswith((".jpg", ".png", ".jpeg"))
])

# -----------------------------
# LOOP
# -----------------------------
for img_name in all_images:
    print("Processing:", img_name)

    img_path = os.path.join(test_folder, img_name)

    hr_img = Image.open(img_path).convert("RGB")
    hr_np = np.array(hr_img)  # keep uint8
    H, W = hr_np.shape[:2]

    # -----------------------------
    # GT
    # -----------------------------
    gt_path = os.path.join(gt_folder, img_name.replace(".jpg", ".txt"))
    gt_boxes, gt_classes = [], []

    if os.path.exists(gt_path):
        with open(gt_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue

                cls = int(parts[0])
                xc, yc, w, h = map(float, parts[1:5])

                x1 = (xc - w / 2) * W
                y1 = (yc - h / 2) * H
                x2 = (xc + w / 2) * W
                y2 = (yc + h / 2) * H

                if cls in UAVDT_TO_VISDRONE:
                    vis = UAVDT_TO_VISDRONE[cls]
                    if vis in VALID_CLASSES:
                        gt_boxes.append([x1, y1, x2, y2])
                        gt_classes.append(VISDRONE_TO_CUSTOM[vis])

    # -----------------------------
    # LR
    # -----------------------------
    lr = cv2.resize(hr_np, (int(W / 3), int(H / 3)))

    # -----------------------------
    # DETECTION (CPU)
    # -----------------------------
    start = time.perf_counter()
    yolo_results_cpu = run_detector(hr_np)
    print("Detections:", 0 if yolo_results_cpu.boxes is None else len(yolo_results_cpu.boxes.xyxy))
    yolo_results_lr = run_detector(lr)
    scale_x = W / lr.shape[1]
    scale_y = H / lr.shape[0]

    if yolo_results_lr.boxes is not None:
        boxes = yolo_results_lr.boxes.xyxy.numpy()
        boxes[:, [0, 2]] *= scale_x
        boxes[:, [1, 3]] *= scale_y
        yolo_results_lr.boxes.xyxy = torch.tensor(boxes)

    runtime_frcnn = time.perf_counter() - start

    if yolo_results_cpu.boxes is None:
        continue

    base_boxes = yolo_results_cpu.boxes.xyxy
    base_classes = yolo_results_cpu.boxes.cls
    num_objects = max(1, len(base_boxes))

    # -----------------------------
    # SR (CPU ONLY)
    # -----------------------------
    lr_tensor = TF.to_tensor(lr)

    start = time.perf_counter()
    with torch.inference_mode():
        sr = sr_tile_infer(sr_model, lr_tensor, tile=96, device="cpu")

    sr = sr.detach().cpu()
    runtime_sr_img = time.perf_counter() - start

    sr_img = sr.squeeze().clamp(0, 1)
    sr_np = (sr_img.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    sr_np = cv2.resize(sr_np, (W, H), interpolation=cv2.INTER_CUBIC)
    sr_results = run_detector(sr_np)
    print("SR boxes:", None if sr_results.boxes is None else len(sr_results.boxes.xyxy))

    del lr_tensor
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # -----------------------------
    # ICRO (UNCHANGED)
    # -----------------------------
    start = time.perf_counter()
    sr_fixed, *_ = detect_sr_icro_fixed(img_path, detector, sr_model, device, save_output=False)
    runtime_icro_fixed_img = time.perf_counter() - start

    start = time.perf_counter()
    sr_adaptive, *_ = detect_sr_icro(img_path, detector, sr_model, device, save_output=False)

    runtime_icro_adapt_img = time.perf_counter() - start

    sr_fixed_results = run_detector(sr_fixed)
    sr_adaptive_results = run_detector(sr_adaptive)

    # -----------------------------
    # OBJECT LOOP
    # -----------------------------
    total_area = H * W

    for i, (box, cls) in enumerate(zip(base_boxes, base_classes)):
        conf_frcnn = get_matching_conf(box, yolo_results_lr)
            
        x1, y1, x2, y2 = map(int, box)
        w, h = x2 - x1, y2 - y1
        if w < 4 or h < 4:
            continue

        size = size_category(w, h)
        gt_class = get_gt_class(box, gt_boxes, gt_classes)

        roi_area = w * h
        area_ratio = roi_area / total_area
        runtime_per_obj = runtime_frcnn / num_objects

        runtime_sr = (runtime_per_obj + runtime_sr_img * area_ratio) / num_objects
        runtime_icro_fixed = (runtime_per_obj + runtime_icro_fixed_img * area_ratio) / num_objects
        runtime_icro_adaptive = (runtime_per_obj + runtime_icro_adapt_img * area_ratio) / num_objects

        conf_sr = get_matching_conf(box, sr_results)
        conf_fixed = get_matching_conf(box, sr_fixed_results)
        conf_adaptive = get_matching_conf(box, sr_adaptive_results)

        psnr_sr, ssim_sr, lpips_sr = compute_object_metrics(sr_np, hr_np, box)
        psnr_fixed, ssim_fixed, lpips_fixed = compute_object_metrics(sr_fixed, hr_np, box)
        psnr_adaptive, ssim_adaptive, lpips_adaptive = compute_object_metrics(sr_adaptive, hr_np, box)

        rows.append({
            "image_name": img_name,
            "object_id": i,
            "size": size,
            "gt_class": gt_class,

            "conf_frcnn": conf_frcnn,
            "conf_sr": conf_sr,
            "conf_icro_fixed": conf_fixed,
            "conf_icro_adaptive": conf_adaptive,

            "runtime_sr": runtime_sr,
            "runtime_icro_fixed": runtime_icro_fixed,
            "runtime_icro_adaptive": runtime_icro_adaptive,

            "psnr_sr": psnr_sr,
            "ssim_sr": ssim_sr,
            "lpips_sr": lpips_sr,

            "psnr_icro_fixed": psnr_fixed,
            "ssim_icro_fixed": ssim_fixed,
            "lpips_icro_fixed": lpips_fixed,

            "psnr_icro_adaptive": psnr_adaptive,
            "ssim_icro_adaptive": ssim_adaptive,
            "lpips_icro_adaptive": lpips_adaptive
        })

    # -----------------------------
    # CLEANUP (SAFE CPU MODE)
    # -----------------------------
    del yolo_results_cpu, sr_np, sr_fixed, sr_adaptive
    gc.collect()

# -----------------------------
# SAVE
# -----------------------------
df = pd.DataFrame(rows)
df.to_excel(excel_path, index=False)

print("Validation complete") # this for frcnn fix

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: c:\Users\Mardyson Justin\Thesis\venv\Lib\site-packages\lpips\weights\v0.1\alex.pth
Processing: DJI-405-720p00331.jpg
Detections: 6
SR boxes: 7

Iteration 1
Avg Scaling Factor   : 1.00
Updated Regions      : 0
Delta Confidence     : 0.0283

Iteration 1
Delta Confidence: 0.0291


C:\Users\Mardyson Justin\AppData\Local\Temp\ipykernel_22800\4189157043.py:108: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  dist = np.sqrt((cx - bx)**2 + (cy - by)**2)


Processing: DJI-405-720p00351.jpg
Detections: 13
SR boxes: 11

Iteration 1
Avg Scaling Factor   : 1.23
Updated Regions      : 0
Delta Confidence     : 0.0961

Iteration 2
Avg Scaling Factor   : 1.15
Updated Regions      : 0
Delta Confidence     : -0.0138

Iteration 1
Delta Confidence: 0.0366
Processing: DJI-405-720p02001.jpg
Detections: 13
SR boxes: 20

Iteration 1
Avg Scaling Factor   : 1.23
Updated Regions      : 0
Delta Confidence     : 0.1110

Iteration 2
Avg Scaling Factor   : 1.17
Updated Regions      : 0
Delta Confidence     : 0.0833

Iteration 3
Avg Scaling Factor   : 1.00
Updated Regions      : 0
Delta Confidence     : 0.0013

Iteration 1
Delta Confidence: 0.1767

Iteration 2
Delta Confidence: 0.0549
Processing: DJI-405-720p02451.jpg
Detections: 10
SR boxes: 10

Iteration 1
Avg Scaling Factor   : 1.30
Updated Regions      : 0
Delta Confidence     : 0.0838

Iteration 2
Avg Scaling Factor   : 1.44
Updated Regions      : 0
Delta Confidence     : 0.0015

Iteration 1
Delta Confiden

In [9]:
# -----------------------------
# FRCNN PER-OBJECT FLOPs (MATCH YOLO STYLE)
# -----------------------------
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
import torchvision.transforms.functional as TF
from thop import profile
from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

device = "cpu"

# -----------------------------
# LOAD MODEL
# -----------------------------
NUM_CLASSES = 11

detector = fasterrcnn_mobilenet_v3_large_fpn(weights=None)
in_features = detector.roi_heads.box_predictor.cls_score.in_features
detector.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)

detector.load_state_dict(torch.load(
    "C:/Users/Mardyson Justin/Thesis/FCNN3/checkpoints/best_model.pth",
    map_location="cpu"
))

detector.to(device)
detector.eval()

# -----------------------------
# SR MODEL
# -----------------------------
sr_model = SRNOInspired()
sr_model.load_state_dict(torch.load(
    "C:/Users/Mardyson Justin/Thesis/SRNO_Checkpoints5/180k_30e/best_model.pth",
    map_location="cpu"
))
sr_model.to(device)
sr_model.eval()

# -----------------------------
# WRAPPERS
# -----------------------------
class FRNNWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        return self.model(x)

class SRWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        _, _, h, w = x.shape
        return self.model(x, h*2, w*2)

frcnn_wrapper = FRNNWrapper(detector)
sr_wrapper = SRWrapper(sr_model)

# -----------------------------
# BASE FLOPs
# -----------------------------
dummy_hr = torch.randn(1, 3, 640, 640)
dummy_lr = torch.randn(1, 3, 224, 224)

flops_frcnn_hr, _ = profile(frcnn_wrapper, inputs=(dummy_hr,), verbose=False)
flops_sr, _ = profile(sr_wrapper, inputs=(dummy_lr,), verbose=False)

NUM_ITER_FIXED = 6
NUM_ITER_ADAPTIVE = 6

# -----------------------------
# DETECTOR FUNCTION
# -----------------------------
def run_detector(img_np):
    detector.eval()

    img_tensor = TF.to_tensor(img_np).to(device)

    with torch.inference_mode():
        outputs = detector([img_tensor])[0]

    boxes = outputs['boxes'].cpu().numpy()
    scores = outputs['scores'].cpu().numpy()

    return boxes, scores

# -----------------------------
# DATASET LOOP
# -----------------------------
test_folder = "C:/Users/Mardyson Justin/Thesis/UAVDT/test/images/"
excel_path = "C:/Users/Mardyson Justin/Thesis/FLOPs/fcnn_per_object_flops.xlsx"

rows = []

for img_name in os.listdir(test_folder):
    if not img_name.lower().endswith((".jpg", ".png", ".jpeg")):
        continue

    print("Processing:", img_name)

    img_path = os.path.join(test_folder, img_name)
    img = np.array(Image.open(img_path).convert("RGB"))

    H, W = img.shape[:2]
    total_area = H * W

    boxes, scores = run_detector(img)

    if boxes is None or len(boxes) == 0:
        continue

    num_objects = max(1, len(boxes))

    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = map(int, box)
        w, h = x2 - x1, y2 - y1

        if w <= 0 or h <= 0:
            continue

        area_ratio = (w * h) / total_area

        # -----------------------------
        # FLOPs PER OBJECT
        # -----------------------------
        frcnn_obj = flops_frcnn_hr / num_objects
        sr_obj = flops_sr / num_objects

        icro_fixed = (
            (flops_frcnn_hr * NUM_ITER_FIXED) / num_objects +
            (flops_sr * area_ratio)
        )

        icro_adaptive = (
            (flops_frcnn_hr * NUM_ITER_ADAPTIVE) / num_objects +
            (flops_sr * area_ratio)
        )

        rows.append({
            "image_name": img_name,
            "object_id": i,
            "width": w,
            "height": h,
            "area_ratio": area_ratio,

            "flops_frcnn": frcnn_obj / 1e9,
            "flops_sr": sr_obj / 1e9,
            "flops_icro_fixed": icro_fixed / 1e9,
            "flops_icro_adaptive": icro_adaptive / 1e9,
        })

# -----------------------------
# SAVE
# -----------------------------
df = pd.DataFrame(rows)
df.to_excel(excel_path, index=False)

print("Saved to:", excel_path)

Processing: det_ucystr__1027.jpg
Processing: det_ucystr__1035.jpg
Processing: det_ucystr__1060.jpg
Processing: det_ucystr__1124.jpg
Processing: det_ucystr__1144.jpg
Processing: det_ucystr__1181.jpg
Processing: det_ucystr__1216.jpg
Processing: det_ucystr__1218.jpg
Processing: det_ucystr__1224.jpg
Processing: det_ucystr__1236.jpg
Processing: det_ucystr__1245.jpg
Processing: det_ucystr__1248.jpg
Processing: det_ucystr__1269.jpg
Processing: det_ucystr__131.jpg
Processing: det_ucystr__1353.jpg
Processing: det_ucystr__1358.jpg
Processing: det_ucystr__1382.jpg
Processing: det_ucystr__1549.jpg
Processing: det_ucystr__1575.jpg
Processing: det_ucystr__1619.jpg
Processing: det_ucystr__171.jpg
Processing: det_ucystr__1760.jpg
Processing: det_ucystr__179.jpg
Processing: det_ucystr__1828.jpg
Processing: det_ucystr__1845.jpg
Processing: det_ucystr__1861.jpg
Processing: det_ucystr__1886.jpg
Processing: det_ucystr__193.jpg
Processing: det_ucystr__202.jpg
Processing: det_ucystr__224.jpg
Processing: det_

In [ ]:
# VRAM-SAFE VALIDATION FOR YOLO
# -----------------------------
import os
import time
import cv2
import torch
import pandas as pd
import numpy as np
from PIL import Image
import torchvision.transforms.functional as TF
from ultralytics import YOLO
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
import lpips
import gc

#withflopss

NUM_CLASSES = 11

# -----------------------------
# DEVICE FOR DETECTOR = CPU (IMPORTANT FIX)
# -----------------------------
device = "cpu"

yolo_model = YOLO("C:/Users/Mardyson Justin/Thesis/yolov9c_visdrone_finetune15/weights/best.pt")  
yolo_model.to("cpu")


# -----------------------------
# SR MODEL (CPU ONLY - IMPORTANT FIX)
# -----------------------------
sr_model = SRNOInspired()
sr_model.load_state_dict(
    torch.load(
        "C:/Users/Mardyson Justin/Thesis/SRNO_Checkpoints5/180k_30e/best_model.pth",
        map_location="cpu"
    )
)
sr_model.to("cpu")
sr_model.eval()

# -----------------------------
# LPIPS (CPU)
# -----------------------------
lpips_model = lpips.LPIPS(net='alex').cpu()
lpips_model.eval()


# -----------------------------
# CONSTANTS
# -----------------------------
VALID_CLASSES = [3, 5, 8]

YOLO_TO_CUSTOM = {
    3: 0,  # car
    5: 1,  # truck
    8: 2   # bus
}
UAVDT_TO_VISDRONE = {0: 3, 1: 5, 2: 8}
VISDRONE_TO_CUSTOM = {3: 0, 5: 1, 8: 2}


# -----------------------------
# DETECTOR FLOPs (RUN ONCE)
# -----------------------------
try:
    sr_flops, sr_params = compute_flops(sr_model, dummy_lr, extra_args=(dummy_lr, 256, 256))
except Exception as e:
    print("SR FLOPs failed:", e)
    sr_flops, sr_params = 0, 0
# -----------------------------
# HELPERS
# -----------------------------
def compute_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    areaA = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    areaB = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    union = areaA + areaB - inter
    return 0 if union == 0 else inter / union


def get_matching_conf(box, results):
    if results.boxes is None:
        return 0

    boxes = results.boxes.xyxy.cpu().numpy()
    confs = results.boxes.conf.cpu().numpy()

    best_score = 0
    x1, y1, x2, y2 = box
    cx = (x1 + x2) / 2
    cy = (y1 + y2) / 2

    for b, c in zip(boxes, confs):
        bx1, by1, bx2, by2 = b

        iou = compute_iou(box, b)

        bx = (bx1 + bx2) / 2
        by = (by1 + by2) / 2
        dist = np.sqrt((cx - bx)**2 + (cy - by)**2)

        # combine both signals instead of hard thresholds
        score = (0.7 * iou + 0.3 * (1 / (1 + dist / 100))) * c

        if score > best_score:
            best_score = score

    return float(best_score)


def get_matching_class(box, results):
    if results.boxes is None:
        return -1

    boxes = results.boxes.xyxy.cpu().numpy()
    classes = results.boxes.cls.cpu().numpy()

    best_iou = 0
    best_cls = -1
    best_dist = 1e9
    best_center_cls = -1

    x1, y1, x2, y2 = box
    cx = (x1 + x2) / 2
    cy = (y1 + y2) / 2

    for b, c in zip(boxes, classes):
        bx1, by1, bx2, by2 = b
        iou = compute_iou(box, b)

        if iou > best_iou:
            best_iou = iou
            best_cls = c

        bx = (bx1 + bx2) / 2
        by = (by1 + by2) / 2
        dist = np.sqrt((cx - bx) ** 2 + (cy - by) ** 2)

        if dist < best_dist:
            best_dist = dist
            best_center_cls = c

    if best_iou > 0.05:
        return int(best_cls)
    if best_dist < 50:
        return int(best_center_cls)
    return -1


def get_gt_class(box, gt_boxes, gt_classes):
    best_iou = 0
    best_cls = -1
    for b, c in zip(gt_boxes, gt_classes):
        iou = compute_iou(box, b)
        if iou > best_iou:
            best_iou = iou
            best_cls = c
    return int(best_cls) if best_iou > 0.5 else -1


def size_category(w, h):
    area = w * h
    if area < 1024:
        return "tiny"
    elif area < 9216:
        return "small"
    return "medium"


def compute_object_metrics(sr_img, hr_img, box):
    x1, y1, x2, y2 = map(int, box)

    H, W = hr_img.shape[:2]

    # clamp
    x1 = max(0, min(x1, W - 1))
    x2 = max(0, min(x2, W - 1))
    y1 = max(0, min(y1, H - 1))
    y2 = max(0, min(y2, H - 1))

    # FIX: invalid or empty crop guard
    if x2 <= x1 or y2 <= y1:
        return 0.0, 0.0, 0.0

    crop_sr = sr_img[y1:y2, x1:x2]
    crop_hr = hr_img[y1:y2, x1:x2]

    # EXTRA SAFETY: empty array guard
    if crop_sr is None or crop_hr is None or crop_sr.size == 0 or crop_hr.size == 0:
        return 0.0, 0.0, 0.0

    # resize safety fallback
    if crop_sr.shape[0] < 8 or crop_sr.shape[1] < 8:
        crop_sr = cv2.resize(crop_sr, (16, 16))
        crop_hr = cv2.resize(crop_hr, (16, 16))

    try:
        psnr = peak_signal_noise_ratio(crop_hr, crop_sr, data_range=255)
    except:
        psnr = 0.0

    try:
        ssim = structural_similarity(
            crop_hr, crop_sr,
            channel_axis=2,
            data_range=255,
            win_size=min(7, crop_hr.shape[0] - 1 if crop_hr.shape[0] % 2 == 0 else crop_hr.shape[0])
        )
    except:
        ssim = 0.0

    try:
        crop_sr_lp = cv2.resize(crop_sr, (64, 64))
        crop_hr_lp = cv2.resize(crop_hr, (64, 64))

        t1 = TF.to_tensor(crop_sr_lp).unsqueeze(0).float() * 2 - 1
        t2 = TF.to_tensor(crop_hr_lp).unsqueeze(0).float() * 2 - 1

        with torch.no_grad():
            lp = lpips_model(t1, t2).item()

    except:
        lp = 0.0

    return psnr, ssim, lp


def sr_tile_infer(sr_model, lr_img, tile=128, device="cpu"):
    """
    Safe SR inference using tiling to avoid VRAM explosion
    """
    _, H, W = lr_img.shape
    sr_output = torch.zeros((3, H * 2, W * 2))  # adjust upscale if needed

    stride = tile
    with torch.inference_mode():
        for y in range(0, H, stride):
            for x in range(0, W, stride):

                patch = lr_img[:, y:y+tile, x:x+tile].unsqueeze(0).to(device)

                # IMPORTANT: force small compute graph
                out = sr_model(patch, patch.shape[-2]*2, patch.shape[-1]*2)

                out = out.squeeze(0).cpu()

                oy, ox = y*2, x*2
                ph, pw = out.shape[-2:]

                sr_output[:, oy:oy+ph, ox:ox+pw] = out

                del patch, out

    return sr_output

# -----------------------------
# RUN DETECTOR (CPU SAFE)
# -----------------------------
def run_detector(img):
    results = yolo_model(img, verbose=False)[0]

    class Result:
        pass

    r = Result()
    r.boxes = None

    if results.boxes is not None and len(results.boxes) > 0:
        class Boxes:
            pass

        r.boxes = Boxes()
        r.boxes.xyxy = results.boxes.xyxy.cpu()
        r.boxes.conf = results.boxes.conf.cpu()
        r.boxes.cls = results.boxes.cls.cpu()

    return r

# -----------------------------
# MAIN LOOP SETUP
# -----------------------------
test_folder = "C:/Users/Mardyson Justin/Thesis/UAVDT/test/images/"
gt_folder = "C:/Users/Mardyson Justin/Thesis/UAVDT/test/labels/"
excel_path = "C:/Users/Mardyson Justin/Thesis/FCNNVal/YOLOTry_VisDrone2019-DET01_Finalized2_results.xlsx"

os.makedirs(os.path.dirname(excel_path), exist_ok=True)

rows = []

all_images = sorted([
    f for f in os.listdir(test_folder)
    if f.lower().endswith((".jpg", ".png", ".jpeg"))
])

# -----------------------------
# LOOP
# -----------------------------
for img_name in all_images:
    print("Processing:", img_name)

    img_path = os.path.join(test_folder, img_name)

    hr_img = Image.open(img_path).convert("RGB")
    hr_np = np.array(hr_img)
    H, W = hr_np.shape[:2]

    # -----------------------------
    # GT
    # -----------------------------
    gt_path = os.path.join(gt_folder, img_name.replace(".jpg", ".txt"))
    gt_boxes, gt_classes = [], []

    if os.path.exists(gt_path):
        with open(gt_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue

                cls = int(parts[0])
                xc, yc, w, h = map(float, parts[1:5])

                x1 = (xc - w / 2) * W
                y1 = (yc - h / 2) * H
                x2 = (xc + w / 2) * W
                y2 = (yc + h / 2) * H

                if cls in UAVDT_TO_VISDRONE:
                    vis = UAVDT_TO_VISDRONE[cls]
                    if vis in VALID_CLASSES:
                        gt_boxes.append([x1, y1, x2, y2])
                        gt_classes.append(VISDRONE_TO_CUSTOM[vis])

    # -----------------------------
    # LR
    # -----------------------------
    lr = cv2.resize(hr_np, (int(W / 3), int(H / 3)))

    # -----------------------------
    # DETECTION (CPU)
    # -----------------------------
    start = time.perf_counter()
    yolo_results_cpu = run_detector(hr_np)
    yolo_results_lr = run_detector(lr)
    scale_x = W / lr.shape[1]
    scale_y = H / lr.shape[0]

    if yolo_results_lr.boxes is not None:
        boxes = yolo_results_lr.boxes.xyxy.numpy()
        boxes[:, [0, 2]] *= scale_x
        boxes[:, [1, 3]] *= scale_y
        yolo_results_lr.boxes.xyxy = torch.tensor(boxes)
    yolo_results_hr = run_detector(hr_np)
    runtime_frcnn = time.perf_counter() - start

    if yolo_results_cpu.boxes is None:
        continue

    base_boxes = yolo_results_cpu.boxes.xyxy.numpy()
    base_classes = yolo_results_cpu.boxes.cls.numpy()
    num_objects = max(1, len(base_boxes))

    # -----------------------------
    # SR (CPU ONLY)
    # -----------------------------
    lr_tensor = TF.to_tensor(lr)

    start = time.perf_counter()
    with torch.inference_mode():
        sr = sr_tile_infer(sr_model, lr_tensor, tile=96, device="cpu")

    sr = sr.detach().cpu()
    runtime_sr_img = time.perf_counter() - start

    sr_img = sr.squeeze().clamp(0, 1)
    sr_np = (sr_img.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    sr_np = cv2.resize(sr_np, (W, H), interpolation=cv2.INTER_CUBIC)
    sr_results = run_detector(sr_np)
    print("SR boxes:", None if sr_results.boxes is None else len(sr_results.boxes.xyxy))

    del lr_tensor
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    # -----------------------------
    # ICRO (UNCHANGED)
    # -----------------------------
    start = time.perf_counter()
    sr_fixed, *_ = detect_sr_icro_fixed_yolo(img_path, save_output=False)
    runtime_icro_fixed_img = time.perf_counter() - start

    start = time.perf_counter()
    sr_adaptive, *_ = detect_sr_icro_yolo(img_path, save_output=False)
    runtime_icro_adapt_img = time.perf_counter() - start

    sr_fixed = cv2.resize(sr_fixed, (W, H), interpolation=cv2.INTER_CUBIC)
    sr_adaptive = cv2.resize(sr_adaptive, (W, H), interpolation=cv2.INTER_CUBIC)

    sr_fixed_results = run_detector(sr_fixed)
    sr_adaptive_results = run_detector(sr_adaptive)

    # -----------------------------
    # OBJECT LOOP
    # -----------------------------
    total_area = H * W

    for i, (box, cls) in enumerate(zip(base_boxes, base_classes)):
        conf_frcnn = get_matching_conf(box, yolo_results_lr)
            
        x1, y1, x2, y2 = map(int, box)
        w, h = x2 - x1, y2 - y1
        if w < 4 or h < 4:
            continue

        size = size_category(w, h)
        gt_class = get_gt_class(box, gt_boxes, gt_classes)

        roi_area = w * h
        area_ratio = roi_area / total_area
        runtime_per_obj = runtime_frcnn / num_objects

        runtime_sr = (runtime_per_obj + runtime_sr_img * area_ratio) / num_objects
        runtime_icro_fixed = (runtime_per_obj + runtime_icro_fixed_img * area_ratio) / num_objects
        runtime_icro_adaptive = (runtime_per_obj + runtime_icro_adapt_img * area_ratio) / num_objects

        conf_sr = get_matching_conf(box, sr_results)
        conf_fixed = get_matching_conf(box, sr_fixed_results)
        conf_adaptive = get_matching_conf(box, sr_adaptive_results)

        psnr_sr, ssim_sr, lpips_sr = compute_object_metrics(sr_np, hr_np, box)
        psnr_fixed, ssim_fixed, lpips_fixed = compute_object_metrics(sr_fixed, hr_np, box)
        psnr_adaptive, ssim_adaptive, lpips_adaptive = compute_object_metrics(sr_adaptive, hr_np, box)

        rows.append({
            "image_name": img_name,
            "object_id": i,
            "size": size,
            "gt_class": gt_class,

            "conf_yolo": conf_frcnn,
            "conf_sr": conf_sr,
            "conf_icro_fixed": conf_fixed,
            "conf_icro_adaptive": conf_adaptive,

            "runtime_sr": runtime_sr,
            "runtime_icro_fixed": runtime_icro_fixed,
            "runtime_icro_adaptive": runtime_icro_adaptive,

            "psnr_sr": psnr_sr,
            "ssim_sr": ssim_sr,
            "lpips_sr": lpips_sr,

            "psnr_icro_fixed": psnr_fixed,
            "ssim_icro_fixed": ssim_fixed,
            "lpips_icro_fixed": lpips_fixed,

            "psnr_icro_adaptive": psnr_adaptive,
            "ssim_icro_adaptive": ssim_adaptive,
            "lpips_icro_adaptive": lpips_adaptive
        })

    # -----------------------------
    # CLEANUP (SAFE CPU MODE)
    # -----------------------------
    del yolo_results_cpu, sr_np, sr_fixed, sr_adaptive
    gc.collect()

# -----------------------------
# SAVE
# -----------------------------
df = pd.DataFrame(rows)
df.to_excel(excel_path, index=False)

print("Validation complete")

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: c:\Users\Mardyson Justin\Thesis\venv\Lib\site-packages\lpips\weights\v0.1\alex.pth
SR FLOPs failed: name 'compute_flops' is not defined
Processing: DJI-405-720p00331.jpg
SR boxes: 1

Iteration 1
Avg Scaling Factor   : 1.67
Updated Regions      : 3
Avg Confidence Before: 0.4001
Avg Confidence After : 0.4977
Delta Confidence     : 0.0976

Iteration 2
Avg Scaling Factor   : 2.33
Updated Regions      : 3
Avg Confidence Before: 0.4977
Avg Confidence After : 0.5388
Delta Confidence     : 0.0411
Converged.

Iteration 1
Avg Scaling Factor   : 2.60
Updated Regions      : 2
Avg Confidence Before: 0.4001
Avg Confidence After : 0.4681
Delta Confidence     : 0.0680

Iteration 2
Avg Scaling Factor   : 2.19
Updated Regions      : 2
Avg Confidence Before: 0.4681
Avg Confidence After : 0.5075
Delta Confidence     : 0.0394
Reached Stable High Confidence.
Processing: DJI-405-720p00351.jpg
SR boxes: 5

Iteration 1


In [ ]:
# -----------------------------
# PER-OBJECT FLOPs EXTRACTION
# -----------------------------
import os
import cv2
import torch
import pandas as pd
import numpy as np
from PIL import Image
from ultralytics import YOLO
from thop import profile

device = "cpu"

# -----------------------------
# LOAD MODELS
# -----------------------------
yolo_model = YOLO("C:/Users/Mardyson Justin/Thesis/yolov9c_visdrone_finetune15/weights/best.pt")
yolo_model.to(device)

# -----------------------------
# WRAPPER
# -----------------------------
class YOLOWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model.model

    def forward(self, x):
        return self.model(x)

yolo_wrapper = YOLOWrapper(yolo_model)

# -----------------------------
# DUMMY INPUTS (SAFE)
# -----------------------------
dummy_hr = torch.randn(1, 3, 640, 640)
dummy_lr = torch.randn(1, 3, 224, 224)

# -----------------------------
# FLOPs (ONCE)
# -----------------------------
flops_yolo_hr, _ = profile(yolo_wrapper, inputs=(dummy_hr,), verbose=False)
flops_yolo_lr, _ = profile(yolo_wrapper, inputs=(dummy_lr,), verbose=False)

# Approx SR FLOPs (reuse LR size)
flops_sr = flops_yolo_lr  # simple approximation (or replace with real SR FLOPs)

# ICRO iterations
NUM_ITER_FIXED = 4
NUM_ITER_ADAPTIVE = 6

# -----------------------------
# PATHS
# -----------------------------
test_folder = "C:/Users/Mardyson Justin/Thesis/UAVDT/test/images/"
excel_path = "C:/Users/Mardyson Justin/Thesis/FLOPs/per_object_flops.xlsx"

rows = []

# -----------------------------
# LOOP
# -----------------------------
for img_name in os.listdir(test_folder):
    if not img_name.lower().endswith((".jpg", ".png", ".jpeg")):
        continue

    print("Processing:", img_name)

    img_path = os.path.join(test_folder, img_name)

    img = Image.open(img_path).convert("RGB")
    img_np = np.array(img)
    H, W = img_np.shape[:2]

    # YOLO detection
    results = yolo_model(img_np, verbose=False)[0]

    if results.boxes is None:
        continue

    boxes = results.boxes.xyxy.cpu().numpy()
    classes = results.boxes.cls.cpu().numpy()

    num_objects = max(1, len(boxes))
    total_area = H * W

    for i, (box, cls) in enumerate(zip(boxes, classes)):
        x1, y1, x2, y2 = map(int, box)
        w, h = x2 - x1, y2 - y1

        if w <= 0 or h <= 0:
            continue

        area_ratio = (w * h) / total_area

        # -----------------------------
        # FLOPs PER OBJECT
        # -----------------------------
        flops_yolo_obj = flops_yolo_hr / num_objects
        flops_sr_obj = flops_sr / num_objects

        flops_icro_fixed_obj = (flops_yolo_hr * NUM_ITER_FIXED) / num_objects + (flops_sr * area_ratio)
        flops_icro_adaptive_obj = (flops_yolo_hr * NUM_ITER_ADAPTIVE) / num_objects + (flops_sr * area_ratio)

        rows.append({
            "image_name": img_name,
            "object_id": i,
            "width": w,
            "height": h,
            "area_ratio": area_ratio,

            "flops_yolo": flops_yolo_obj / 1e9,
            "flops_sr": flops_sr_obj / 1e9,
            "flops_icro_fixed": flops_icro_fixed_obj / 1e9,
            "flops_icro_adaptive": flops_icro_adaptive_obj / 1e9,
        })

# -----------------------------
# SAVE
# -----------------------------
df = pd.DataFrame(rows)
os.makedirs(os.path.dirname(excel_path), exist_ok=True)
df.to_excel(excel_path, index=False)

print("Saved to:", excel_path)

Processing: det_ucystr__1027.jpg
Processing: det_ucystr__1035.jpg
Processing: det_ucystr__1060.jpg
Processing: det_ucystr__1124.jpg
Processing: det_ucystr__1144.jpg
Processing: det_ucystr__1181.jpg
Processing: det_ucystr__1216.jpg
Processing: det_ucystr__1218.jpg
Processing: det_ucystr__1224.jpg
Processing: det_ucystr__1236.jpg
Processing: det_ucystr__1245.jpg
Processing: det_ucystr__1248.jpg
Processing: det_ucystr__1269.jpg
Processing: det_ucystr__131.jpg
Processing: det_ucystr__1353.jpg
Processing: det_ucystr__1358.jpg
Processing: det_ucystr__1382.jpg
Processing: det_ucystr__1549.jpg
Processing: det_ucystr__1575.jpg
Processing: det_ucystr__1619.jpg
Processing: det_ucystr__171.jpg
Processing: det_ucystr__1760.jpg
Processing: det_ucystr__179.jpg
Processing: det_ucystr__1828.jpg
Processing: det_ucystr__1845.jpg
Processing: det_ucystr__1861.jpg
Processing: det_ucystr__1886.jpg
Processing: det_ucystr__193.jpg
Processing: det_ucystr__202.jpg
Processing: det_ucystr__224.jpg
Processing: det_

In [8]:
validate(
    test_dir="C:/Users/Mardyson Justin/Thesis/UAVDT/test/images/",
    detector=detector,
    sr_model=sr_model,
    icro_fixed_fn=detect_sr_icro_fixed,
    icro_adapt_fn=detect_sr_icro,
    output_excel="C:/Users/Mardyson Justin/Thesis/FCNNVal/FCNNTest_UAVDT1_Finalized_results.xlsx"
)

NameError: name 'compute_frcnn_flops' is not defined